# Kaggle Notebook Pipeline

**Run on [kaggle.com](https://www.kaggle.com) only** — not on your laptop.

1. **Settings → Internet → On** (needed for `git clone` + `pip install`)
2. **Settings → Accelerator → None** (CPU is enough for Phase 1)
3. **Add Data** → attach input dataset(s) listed in the config cell below
4. Run all cells, then **Save Version** → **Save as Dataset** to pass `data/` to the next notebook

**Inputs:** `RAW_EEG_INPUT` only.


In [ ]:
# --- Kaggle configuration (edit slugs to match your input datasets) ---
REPO_URL = "https://github.com/RandomPerson5571/ad_eeg.git"
REPO_BRANCH = "main"
PROJECT_DIR = "/kaggle/working/ad_eeg"

# Kaggle dataset slug with raw EEG (must contain EEG_data/dataset2/ and dataset3/)
RAW_EEG_INPUT = "REPLACE_WITH_RAW_EEG_DATASET_SLUG"

# Optional: output from a prior pipeline notebook (must contain data/ at root)
PIPELINE_INPUT = None  # e.g. "REPLACE_WITH_PRIOR_PIPELINE_OUTPUT_SLUG"


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError(
        "This notebook runs on Kaggle only. "
        "Upload to kaggle.com, enable Internet, attach input datasets, then run."
    )

PROJECT_DIR = Path(PROJECT_DIR)


def run(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if not PROJECT_DIR.exists():
    run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=PROJECT_DIR)
print(f"Project root: {PROJECT_DIR.resolve()}", flush=True)


def _find_eeg_root(slug: str) -> Path | None:
    base = Path("/kaggle/input") / slug
    if not base.exists():
        return None
    if (base / "EEG_data").is_dir():
        return base / "EEG_data"
    if (base / "dataset2").is_dir():
        return base
    for child in base.iterdir():
        if child.is_dir() and (child / "EEG_data").is_dir():
            return child / "EEG_data"
        if child.is_dir() and (child / "dataset2").is_dir():
            return child
    return None


eeg_link = PROJECT_DIR / "EEG_data"
if RAW_EEG_INPUT:
    src = _find_eeg_root(RAW_EEG_INPUT)
    if src is None:
        raise FileNotFoundError(
            f"Raw EEG not found for slug '{RAW_EEG_INPUT}'. "
            "Add Data → your dataset with EEG_data/dataset2/ and dataset3/."
        )
    if eeg_link.is_symlink():
        eeg_link.unlink()
    elif eeg_link.is_dir() and not eeg_link.is_symlink():
        pass
    elif eeg_link.exists():
        eeg_link.unlink()
    if not eeg_link.exists():
        os.symlink(src, eeg_link)
    print(f"EEG_data → {src}", flush=True)

if PIPELINE_INPUT:
    pipeline_src = Path("/kaggle/input") / PIPELINE_INPUT / "data"
    if not pipeline_src.exists():
        pipeline_src = Path("/kaggle/input") / PIPELINE_INPUT
        if not (pipeline_src / "preprocessed").exists() and not (pipeline_src / "audit").exists():
            raise FileNotFoundError(
                f"Pipeline input '{PIPELINE_INPUT}' has no data/ folder. "
                "Save the previous notebook version as a Dataset first."
            )
    dest = PROJECT_DIR / "data"
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(pipeline_src, dest, dirs_exist_ok=True)
    print(f"Restored pipeline data from {pipeline_src}", flush=True)


# 01 — Preprocessing

Mode-based preprocessing pipeline for fast iteration and production runs.

| Mode | Purpose | Output |
|------|---------|--------|
| **inspect** | One subject, interactive QC plots, debugging | `/kaggle/working/test_output/inspect/` |
| **test** | First N subjects, validation metrics, regression check | `/kaggle/working/test_output/test/` |
| **full** | Entire dataset, persisted checkpoints | `data/` → publish as Kaggle Dataset |

**Flow:** Configuration → Environment setup → Load config → Branch on `MODE` → (full only) publish artifacts.


In [ ]:
# --- Pipeline configuration (edit before running) ---
MODE = "test"          # "inspect" | "test" | "full"
DATASET = "dataset2"   # "dataset2" | "dataset3" | "all"
EXPERIMENT = "baseline"
FORCE = False
WORKERS = 2
TEST_SUBJECTS = 5      # used when MODE == "test"
INSPECT_SUBJECT = 1    # subject number when MODE == "inspect"

VALID_MODES = {"inspect", "test", "full"}
if MODE not in VALID_MODES:
    raise ValueError(f"MODE must be one of {VALID_MODES}, got {MODE!r}")

TEST_OUTPUT = Path("/kaggle/working/test_output")


In [ ]:
from eeg.config import load_experiment, resolve_dataset
from eeg.repro import init_repro, snapshot_environment

dataset_specs = resolve_dataset(DATASET)
config = load_experiment(EXPERIMENT)

CONFIG = {
    "mode": MODE,
    "dataset": DATASET,
    "datasets": [ds.name for ds in dataset_specs],
    "experiment": EXPERIMENT,
    "force": FORCE,
    "workers": WORKERS,
    "test_subjects": TEST_SUBJECTS,
    "inspect_subject": INSPECT_SUBJECT,
    "seed": config.get("training", {}).get("random_state", 42),
}
repro = init_repro(CONFIG["seed"])
env = snapshot_environment()
print(CONFIG)


In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

from eeg.config import experiment_metadata
from eeg.io import write_json
from eeg.paths import qc_report_dir
from eeg.preprocess_report import write_preprocess_report
from eeg.qc import BAND_RANGES, backfill_spectral_qc, preprocessing_metrics
from eeg.runner import summarize_batch
from eeg.visualization import plot_preprocessing_panels_from_checkpoints
from eeg.cli import subject_num_from_id
from scripts.preprocess_dataset import run_preprocess

QC_PLOTS = False  # set True for per-subject PNGs in full/test mode


def _subject_nums(ds):
    if MODE == "inspect":
        return [INSPECT_SUBJECT]
    return list(range(1, TEST_SUBJECTS + 1))


def _spectral_summary(metrics: dict) -> dict:
    """Extract µV²/Hz band-power fields for notebook summaries."""
    out = {}
    for band in BAND_RANGES:
        if metrics.get(f"{band}_before_uv2") is not None:
            out[f"{band}_before_uv2"] = metrics[f"{band}_before_uv2"]
        if metrics.get(f"{band}_delta_uv2") is not None:
            out[f"{band}_delta_uv2"] = metrics[f"{band}_delta_uv2"]
    return out


def _refresh_dataset_qc(ds, limit=None):
    """Backfill spectral QC from checkpoints and regenerate summary.csv."""
    patched = backfill_spectral_qc(ds.name, EXPERIMENT, limit=limit)
    report_paths = write_preprocess_report(
        ds.name, EXPERIMENT, config=config, qc_plots=QC_PLOTS, dataset_spec=ds
    )
    if patched:
        print(f"  [{ds.name}] backfilled spectral QC for {len(patched)} subject(s)")
    print(f"  [{ds.name}] QC report → {report_paths['summary_csv']}")
    return report_paths


def _qc_subject(ds, subject_num, out_dir):
    metrics = preprocessing_metrics(ds, subject_num, EXPERIMENT)
    plot_path = plot_preprocessing_panels_from_checkpoints(
        ds, subject_num, EXPERIMENT, out_dir
    )
    alpha = metrics.get("alpha_before_uv2")
    alpha_d = metrics.get("alpha_delta_uv2")
    spectral = (
        f" alpha={alpha:.2f}µV² Δ={alpha_d:+.2f}"
        if alpha is not None and alpha_d is not None
        else ""
    )
    print(
        f"  {metrics['participant_id']}: bad_ch={metrics['n_bad_channels']} "
        f"rejected={metrics['n_epochs_rejected']}/{metrics['n_epochs_before_ar']}"
        f"{spectral} → {plot_path.name}"
    )
    return metrics, plot_path


summary = {
    "mode": MODE,
    "dataset": DATASET,
    "experiment": EXPERIMENT,
    "force": FORCE,
    "started_at": datetime.now(timezone.utc).isoformat(),
}
t0 = time.perf_counter()

if MODE == "inspect":
    out_root = TEST_OUTPUT / "inspect"
    out_root.mkdir(parents=True, exist_ok=True)
    summary["subjects"] = []

    for ds in dataset_specs:
        ds_out = out_root / ds.name
        ds_out.mkdir(parents=True, exist_ok=True)
        print(f"\n[{ds.name}] preprocess + inspect subject {INSPECT_SUBJECT}")
        run_preprocess(
            ds.name,
            EXPERIMENT,
            workers=1,
            force=FORCE,
            limit=None,
            subject=f"sub-{INSPECT_SUBJECT:03d}",
            qc_plots=True,
        )
        _refresh_dataset_qc(ds, limit=INSPECT_SUBJECT)
        metrics, _ = _qc_subject(ds, INSPECT_SUBJECT, ds_out)
        summary["subjects"].append({**metrics, **_spectral_summary(metrics)})
        summary["qc_report"] = str(qc_report_dir(ds.name, EXPERIMENT) / "summary.csv")

elif MODE == "test":
    out_root = TEST_OUTPUT / "test"
    out_root.mkdir(parents=True, exist_ok=True)
    summary["datasets"] = {}

    for ds in dataset_specs:
        ds_out = out_root / ds.name
        ds_out.mkdir(parents=True, exist_ok=True)
        print(f"\n[{ds.name}] preprocessing first {TEST_SUBJECTS} subjects...")
        results = run_preprocess(
            ds.name,
            EXPERIMENT,
            workers=WORKERS,
            force=FORCE,
            limit=TEST_SUBJECTS,
            qc_plots=QC_PLOTS,
        )
        report_paths = _refresh_dataset_qc(ds, limit=TEST_SUBJECTS)
        batch = summarize_batch(results)
        summary["datasets"][ds.name] = {
            "completed": batch.completed,
            "skipped": batch.skipped,
            "failed": batch.failed,
            "qc_report": str(report_paths["summary_csv"]),
            "subjects": [
                {
                    "participant_id": r.log.get("participant_id"),
                    "status": r.status,
                    "runtime_seconds": r.log.get("runtime_seconds"),
                    "n_bad_channels": len(r.log.get("bad_channels", [])),
                    "n_epochs_rejected": r.log.get("n_epochs_rejected"),
                    **_spectral_summary(
                        preprocessing_metrics(
                            ds,
                            subject_num_from_id(r.log.get("participant_id")),
                            EXPERIMENT,
                        )
                    ),
                }
                for r in results
            ],
        }
        print(
            f"[{ds.name}] completed={batch.completed} "
            f"skipped={batch.skipped} failed={batch.failed}"
        )

        print(f"[{ds.name}] QC plots...")
        for sn in _subject_nums(ds):
            _qc_subject(ds, sn, ds_out)

elif MODE == "full":
    summary["datasets"] = {}
    all_results = []

    for ds in dataset_specs:
        print(f"\n[{ds.name}] full preprocessing run...")
        results = run_preprocess(
            ds.name,
            EXPERIMENT,
            workers=WORKERS,
            force=FORCE,
            limit=None,
            qc_plots=QC_PLOTS,
        )
        report_paths = _refresh_dataset_qc(ds)
        batch = summarize_batch(results)
        all_results.extend(results)
        summary["datasets"][ds.name] = {
            "completed": batch.completed,
            "skipped": batch.skipped,
            "failed": batch.failed,
            "n_subjects": len(results),
            "qc_report": str(report_paths["summary_csv"]),
        }
        print(
            f"[{ds.name}] completed={batch.completed} "
            f"skipped={batch.skipped} failed={batch.failed}"
        )

    runtimes = [r.log.get("runtime_seconds", 0) for r in all_results if r.log.get("runtime_seconds")]
    summary["mean_runtime_seconds"] = round(sum(runtimes) / len(runtimes), 2) if runtimes else None
    summary["config"] = experiment_metadata(
        DATASET, EXPERIMENT, config, n_processed=len(all_results)
    )

else:
    raise ValueError(f"Unknown MODE: {MODE}")

summary["elapsed_seconds"] = round(time.perf_counter() - t0, 2)
summary["finished_at"] = datetime.now(timezone.utc).isoformat()

if MODE == "full":
    meta_path = Path("data") / "preprocess_full_summary.json"
else:
    meta_path = TEST_OUTPUT / f"preprocess_{MODE}_summary.json"
meta_path.parent.mkdir(parents=True, exist_ok=True)
write_json(meta_path, summary)
print(f"\nSummary → {meta_path}")
print(json.dumps(summary, indent=2, default=str))

In [ ]:
# Publish artifacts for the next notebook (full mode only)
if MODE == "full" and Path("/kaggle/input").exists():
    out = Path("/kaggle/working/pipeline_output")
    src = Path("data")
    if src.exists():
        shutil.copytree(src, out / "data", dirs_exist_ok=True)
        print(f"Output ready: {out}")
        print("Save Version → Save output as new Kaggle Dataset, then attach in the next notebook.")
elif MODE != "full":
    print(f"MODE={MODE!r}: skipping Kaggle dataset publish (use MODE='full' for production output).")
